In [0]:
# Reference the bronze table created by the data profiling step
TABLE_NAME = "students_data.`team4-chicago`.food_inspections_bronze"

# Display current schema before enforcement
print("Current schema (before enforcement):")
spark.table(TABLE_NAME).printSchema()

In [0]:
# Enforce NOT NULL constraints (expects null rows already cleaned by data profiling step)
not_null_columns = [
    "Inspection_ID", "DBA_Name", "License", "Facility_Type",
    "Risk", "Address", "City", "State", "Zip",
    "Inspection_Date", "Inspection_Type", "Results"
]

for col_name in not_null_columns:
    spark.sql(f"ALTER TABLE {TABLE_NAME} ALTER COLUMN {col_name} SET NOT NULL")
    print(f"SET NOT NULL: {col_name}")

print(f"\nAll {len(not_null_columns)} NOT NULL constraints enforced.")
print(f"\nFinal schema:")
spark.table(TABLE_NAME).printSchema()

In [0]:
%pip install ipytest -q

In [0]:
import pytest
import ipytest
from pyspark.sql.types import IntegerType, StringType, DateType, DoubleType

ipytest.autoconfig()

# Self-contained: define table name and load schema fresh
TABLE_NAME = "students_data.`team4-chicago`.food_inspections_bronze"
_df = spark.table(TABLE_NAME)
_schema = {field.name: field for field in _df.schema.fields}

# ---- Data Type Tests ----

@pytest.mark.parametrize("col_name, expected_type", [
    ("Inspection_ID", IntegerType()),
    ("DBA_Name", StringType()),
    ("AKA_Name", StringType()),
    ("License", IntegerType()),
    ("Facility_Type", StringType()),
    ("Risk", StringType()),
    ("Address", StringType()),
    ("City", StringType()),
    ("State", StringType()),
    ("Zip", IntegerType()),
    ("Inspection_Date", DateType()),
    ("Inspection_Type", StringType()),
    ("Results", StringType()),
    ("Violations", StringType()),
    ("Latitude", DoubleType()),
    ("Longitude", DoubleType()),
    ("Location", StringType()),
])
def test_column_data_type(col_name, expected_type):
    assert col_name in _schema, f"Column '{col_name}' not found in table"
    assert _schema[col_name].dataType == expected_type, (
        f"{col_name}: expected {expected_type}, got {_schema[col_name].dataType}"
    )

# ---- NOT NULL Constraint Tests ----

@pytest.mark.parametrize("col_name", [
    "Inspection_ID", "DBA_Name", "License", "Facility_Type",
    "Risk", "Address", "City", "State", "Zip",
    "Inspection_Date", "Inspection_Type", "Results",
])
def test_column_is_not_nullable(col_name):
    assert _schema[col_name].nullable is False, (
        f"{col_name}: expected nullable=False, got nullable=True"
    )

# ---- Nullable Column Tests ----

@pytest.mark.parametrize("col_name", [
    "AKA_Name", "Violations", "Latitude", "Longitude", "Location",
])
def test_column_is_nullable(col_name):
    assert _schema[col_name].nullable is True, (
        f"{col_name}: expected nullable=True, got nullable=False"
    )

# ---- Table Not Empty ----

def test_table_is_not_empty():
    assert _df.count() > 0, "Table is empty"

# ---- No Nulls in NOT NULL Columns ----

@pytest.mark.parametrize("col_name", [
    "Inspection_ID", "DBA_Name", "License", "Facility_Type",
    "Risk", "Address", "City", "State", "Zip",
    "Inspection_Date", "Inspection_Type", "Results",
])
def test_no_nulls_in_required_column(col_name):
    null_count = _df.filter(_df[col_name].isNull()).count()
    assert null_count == 0, (
        f"{col_name}: found {null_count} null rows in NOT NULL column"
    )

ipytest.run("-v")